# Module 0 — The Evaluation Landscape

**Where this fits:** this is Module 0 of the *RAG, Agent & Tool Evaluation* tutorial (see the folder `README.md` for the full table of contents). Everything after this notebook is hands-on — real metrics, real code, real API calls. This one is deliberately the exception: no API keys, no cost, nothing to run except a couple of reference tables. Its only job is to give you a map before you start walking, because the material this tutorial pulls together was originally scattered across four different tools (DeepEval, RAGAS, MLflow, Arize Phoenix) with no shared vocabulary between them. Read this once, then use it as a lookup table as you go.

**By the end of this notebook you'll be able to answer:**
- What are the different *kinds* of evaluation a RAG/agent system needs, and how do they relate to each other?
- What does "reference-based" vs "referenceless" actually mean, and why does it matter for cost and scalability?
- What is "LLM-as-judge," and what are its actual failure modes — not just "it's an LLM, it might be wrong"?
- Given a tool (DeepEval / RAGAS / MLflow / Arize Phoenix), what is it *for*, and where in this tutorial do you see it?
- Given a question you actually have ("is my agent calling the right tools?"), which module and which metric answers it?

## 1. The taxonomy: what are you actually evaluating?

"Evaluate the RAG system" or "evaluate the agent" is not one task — it's a stack of narrower questions, each with its own failure modes and its own metrics. Getting the taxonomy straight up front is what stops you from, say, using a retrieval metric to diagnose a generation bug (a faithfulness failure can look identical to a retrieval failure from the outside — the answer is wrong either way — but the fix is completely different).

### 1a. By pipeline stage (RAG-specific)

| Stage | Question it answers | Failure it catches | Example metrics |
|---|---|---|---|
| **Retrieval** | Did we fetch the right chunks? | Retriever returns irrelevant, missing, or badly-ranked context | Precision@K, Recall@K, MRR, nDCG, `ContextualPrecisionMetric`, `ContextualRecallMetric`, `ContextualRelevancyMetric` |
| **Generation (referenceless)** | Given what we retrieved, is the answer grounded and on-topic? | Hallucination on top of correct context; answering a different question than the one asked | `FaithfulnessMetric`, `AnswerRelevancyMetric`, RAGAS Faithfulness |
| **Generation (reference-based)** | Is the answer actually *correct*, against a known-good answer? | Grounded-but-wrong-because-the-context-itself-was-wrong; wording drift from the expected answer | Answer Correctness (custom `GEval`), Answer Semantic Similarity (embeddings) |
| **End-to-end** | Does the whole pipeline, run as a system, hold up against a labeled dataset? | Any of the above, measured in aggregate over many queries instead of one | A synthetic golden dataset + the full metric suite above, batched |

### 1b. By behavior (agent-specific)

| Behavior | Question it answers | Failure it catches | Example metrics |
|---|---|---|---|
| **Tool selection** | Did the agent call the right tools? | Wrong tool, missing tool, tool called when none was needed | `ToolCorrectnessMetric` (reference-based), Tool Selection Accuracy (trace-based, coverage-only) |
| **Argument correctness** | Were the *arguments* passed to each tool correct? | Right tool, malformed or wrong arguments — a failure mode tool-selection checks can't see | `ArgumentCorrectnessMetric`, deterministic schema validation |
| **Trajectory** | Across a multi-step run, was the *sequence* of actions sound? | Redundant calls, wasted steps, no recovery from an error, right destination via a wasteful path | Step-wise Accuracy, Redundant-Call detection, Recovery Rate, fuzzy Trajectory Match |
| **Task completion** | Forget the mechanics — did the agent actually accomplish what the user asked? | Every tool call "correct" in isolation, but a stated constraint (budget, deadline, scope) silently dropped | `TaskCompletionMetric`, End-State Verification, custom `GEval` rubric |
| **Conversational quality** | Across a multi-turn dialogue, does the agent stay relevant, remember prior turns, and stay in its role/safety bounds? | Forgetting earlier context, drifting off-topic turn by turn, giving advice it shouldn't | `TurnRelevancyMetric`, `KnowledgeRetentionMetric`, `ConversationalGEval` |

Notice the shape of the agent-side taxonomy: it goes from *mechanics* (did it call the right tool, with the right arguments) to *process* (was the sequence of steps sound) to *outcome* (did it actually get the job done). A system can pass every mechanics-level check and still fail at the outcome level — that's the whole reason task-completion and end-state verification exist as separate concerns from tool correctness, not a redundant restatement of it.

## 2. Reference-based vs. referenceless — and why it's the axis that decides your cost

Every metric above falls into one of two camps, and this single distinction drives most of the practical decisions you'll make about *when* to use which metric:

- **Reference-based** metrics compare the system's output against a known-correct answer (`expected_output`, `expected_tools`, a gold trajectory). They're precise and great for regression testing / CI, but **someone has to write and maintain the reference** — which means they don't scale to arbitrary production traffic, only to curated test sets.
- **Referenceless** metrics judge quality *without* a ground truth — typically by checking internal consistency (is the answer grounded in the context that was actually retrieved?) or by using an LLM to judge quality directly against a rubric. They scale to any input with zero labeling cost, at the price of being less precise and dependent on judge-model quality.

**The practical rule:** use reference-based metrics in development and CI, where you control the test set and want a hard pass/fail bar. Use referenceless metrics in production monitoring, where you can't hand-label every real user query but still want a quality signal. Most of the mature setups in this tutorial run *both*, at different points in the lifecycle — not as a choice between them.

## 3. LLM-as-judge: what it actually is, and where it actually breaks

Most of the referenceless metrics above (and some reference-based ones, like Answer Correctness) work by asking an LLM to *judge* the output — usually against a structured rubric, sometimes extracting intermediate claims first (e.g. faithfulness checks often decompose the answer into individual factual claims, then check each one against the retrieved context, rather than judging the whole answer as one blob).

This is powerful — it's the only practical way to score open-ended text at all — but it is not a free "ground truth oracle," and treating it like one is the single most common mistake in eval work. Concretely, watch for:

- **Judge-model choice matters.** A weaker judge model produces noisier, less reliable scores — the judge needs to be at least as capable as the task requires, sometimes more capable than the system being judged.
- **Non-determinism.** Two runs of the same judge on the same input can disagree, especially near the threshold. Don't treat a single `metric.score` as exact; treat it as a noisy estimate, and set thresholds with that in mind.
- **Prompt/criteria sensitivity.** A vaguely-worded `GEval` criteria string produces a vague, inconsistent judge. The specificity you put into the rubric is the specificity you get back in the judgment — this is why several notebooks in this tutorial write quite detailed, example-anchored criteria rather than one-line prompts.
- **Cost and latency compound.** Every LLM-judged metric is (at least) one extra API call per test case, per metric. A suite of 5 metrics over 500 test cases is 2,500+ judge calls — this is *why* the reference-based-vs-referenceless split above matters operationally, not just conceptually.
- **It can be gamed by fluent-but-wrong output.** An LLM judge reads text and reasons about whether it *sounds* right. This is exactly why Module 3 (agent trajectory) and Module 5 (the CrewAI capstone) put weight on **end-state / outcome verification** — checking the real system of record directly — as the strongest signal available, precisely because it can't be fooled by a confident sentence that misrepresents what actually happened.

## 4. The tooling landscape

This tutorial deliberately keeps four different tools in view rather than picking one, because the source material already used all four, and because in practice you'll encounter systems built on each of them. Knowing what each is *for* is more useful than picking a favorite.

| Tool | What it's really for | Where you'll see it in this tutorial |
|---|---|---|
| **DeepEval** | The general-purpose metric library this tutorial leans on most — dozens of pre-built metrics (retrieval, generation, tool use, task completion, conversational) plus `GEval` for custom LLM-judge rubrics. Pytest-friendly, good for CI. | Modules 1-3 (primary framework throughout) |
| **RAGAS** | RAG-specific metric library, older and narrower in scope than DeepEval but a common default in the RAG ecosystem — worth knowing on its own terms, and useful as a second opinion alongside DeepEval on the same test case. | Module 1 (shown side-by-side with DeepEval on generator metrics; also a small standalone quickstart) |
| **MLflow (`mlflow.evaluate`)** | Evaluation as part of an experiment-tracking platform — the same conceptual metrics as DeepEval, but versioned, logged, and comparable across runs the way you'd compare model training experiments. The right choice if your team already lives in MLflow for everything else. | Module 2 (the same 3 questions from DeepEval, reimplemented, as a direct framework comparison) |
| **Arize Phoenix** | Production-grade **tracing** plus an **offline experiment framework** — lets you instrument a live agent, capture every span (tool call, LLM call, retrieval), then run structured evaluators against captured traces and compare prompt/config versions as formal experiments. The heaviest-weight tool here, and the only one built around live tracing rather than one-off test cases. | Module 4 (the 5-lab progressive course) |

None of these are "the right one" in the abstract — DeepEval and RAGAS answer "is this output good," MLflow answers "is this output good, tracked as part of my experiment history," and Phoenix answers "is this output good, and here's the full trace of *how* the system produced it." A mature setup typically uses more than one, for different purposes, exactly as this tutorial's modules do.

In [1]:
# ============ QUICK-REFERENCE LOOKUP ============
# A plain-Python lookup table for "I have this question -> which module/metric answers it?"
# No API calls, nothing to configure -- just a map you can come back to.

EVAL_QUESTION_MAP = [
    {
        "question": "Did the retriever fetch the right chunks?",
        "module": "Module 1 (RAG Evaluation)",
        "metrics": ["Precision@K", "Recall@K", "MRR", "nDCG", "ContextualPrecisionMetric",
                    "ContextualRecallMetric", "ContextualRelevancyMetric"],
    },
    {
        "question": "Is the generated answer grounded in what was retrieved (no hallucination)?",
        "module": "Module 1 (RAG Evaluation)",
        "metrics": ["FaithfulnessMetric", "RAGAS Faithfulness"],
    },
    {
        "question": "Is the generated answer actually correct against a known answer?",
        "module": "Module 1 (RAG Evaluation)",
        "metrics": ["Answer Correctness (custom GEval)", "Answer Semantic Similarity (embeddings)"],
    },
    {
        "question": "Did the agent call the right tools, with the right arguments?",
        "module": "Module 2 (Tool & Task Evaluation)",
        "metrics": ["ToolCorrectnessMetric", "ArgumentCorrectnessMetric"],
    },
    {
        "question": "Did the agent actually accomplish what the user asked for?",
        "module": "Module 2 (Tool & Task Evaluation)",
        "metrics": ["TaskCompletionMetric", "custom GEval rubric", "end-state verification"],
    },
    {
        "question": "Does the agent stay coherent and relevant across a multi-turn conversation?",
        "module": "Module 2 (Tool & Task Evaluation)",
        "metrics": ["TurnRelevancyMetric", "KnowledgeRetentionMetric", "ConversationalGEval"],
    },
    {
        "question": "Across a multi-step run, was the sequence of tool calls efficient and self-correcting?",
        "module": "Module 3 (Agent Trajectory Evaluation)",
        "metrics": ["Step-wise Accuracy", "Redundant-Call detection", "Recovery Rate",
                    "fuzzy Trajectory Match"],
    },
    {
        "question": "How do I evaluate a live, traced, production agent with real experiments?",
        "module": "Module 4 (Production Tracing & Experimentation -- Arize Phoenix)",
        "metrics": ["router/skill LLM-judge evals", "Phoenix experiment-based trajectory evals",
                    "structured multi-evaluator comparisons"],
    },
    {
        "question": "How does this all look end-to-end, on a real running multi-agent app?",
        "module": "Module 5 (Capstone -- CrewAI Travel Planner)",
        "metrics": ["TaskCompletionMetric against a live FastAPI backend, batch-scored from Excel"],
    },
]

for row in EVAL_QUESTION_MAP:
    print(f"Q: {row['question']}")
    print(f"   -> {row['module']}")
    print(f"   -> metrics: {', '.join(row['metrics'])}")
    print()

Q: Did the retriever fetch the right chunks?
   -> Module 1 (RAG Evaluation)
   -> metrics: Precision@K, Recall@K, MRR, nDCG, ContextualPrecisionMetric, ContextualRecallMetric, ContextualRelevancyMetric

Q: Is the generated answer grounded in what was retrieved (no hallucination)?
   -> Module 1 (RAG Evaluation)
   -> metrics: FaithfulnessMetric, RAGAS Faithfulness

Q: Is the generated answer actually correct against a known answer?
   -> Module 1 (RAG Evaluation)
   -> metrics: Answer Correctness (custom GEval), Answer Semantic Similarity (embeddings)

Q: Did the agent call the right tools, with the right arguments?
   -> Module 2 (Tool & Task Evaluation)
   -> metrics: ToolCorrectnessMetric, ArgumentCorrectnessMetric

Q: Did the agent actually accomplish what the user asked for?
   -> Module 2 (Tool & Task Evaluation)
   -> metrics: TaskCompletionMetric, custom GEval rubric, end-state verification

Q: Does the agent stay coherent and relevant across a multi-turn conversation?
   -> M

## 5. How this tutorial is organized

| Module | Answers | Primary tool(s) |
|---|---|---|
| **0 — Evaluation Landscape** *(this notebook)* | What kinds of evaluation exist, and how do the tools relate? | — |
| **1 — RAG Evaluation** | Retrieval quality, generation quality (with and without reference), evaluation wired into the pipeline itself, and a full build-a-RAG-system-then-evaluate-it capstone | DeepEval, RAGAS |
| **2 — Conversational, Tool & Task Evaluation** | Multi-turn conversational quality, tool-call correctness, task completion — then the same three questions again through MLflow, as a framework comparison | DeepEval, MLflow |
| **3 — Agent Trajectory Evaluation** | Step-by-step evaluation of a full multi-tool agent run, mocked then with a real tool-calling agent, plus a 9-metric deep dive (tool selection, arguments, redundancy, step accuracy, task success, trajectory match, end-state verification, recovery, cost proxy) | DeepEval-adjacent, hand-rolled trace metrics |
| **4 — Production Tracing & Experimentation** | How this looks with real tracing infrastructure and a formal offline-experiment framework, via a 5-lab progressive course (build → trace → router/skill evals → trajectory evals → structured evals) | Arize Phoenix |
| **5 — Capstone: Real Multi-Agent System Evaluation** | Everything above, applied to a real running CrewAI multi-agent app | DeepEval |

**If you only have an hour:** Module 0 (this notebook) → Module 1's LLM-judged retrieval notebook → Module 2's tool-use notebook → Module 3's trajectory deep dive. That sequence hits retrieval, generation, tool-mechanics, and outcome-level evaluation — the four ideas that generalize to almost any RAG or agent system you'll actually be asked to evaluate.

**A note on the tooling caveat carried through this whole tutorial:** the source notebooks this tutorial draws from were pinned to different `deepeval` versions (`3.1.0` through `3.8.8` in various subfolders). This tutorial targets `deepeval==4.1.8` (the repo root `.venv`) throughout — every metric class named above is confirmed present in that version. If you're running an individual source notebook directly instead of this tutorial's copies, check its own folder's pin first.